In [37]:
import torch
from torch import nn
import torch.nn.functional as F

In [38]:
# dmodel is the size of the weight matrices
# if dmodel is 2 we have a 2 word embedding value

class SelfAttention(nn.Module):
    def __init__(self, d_model=2, row_dim=0, col_dim=1):
        super().__init__()
        self.d_model = d_model
        self.row_dim = row_dim
        self.col_dim = col_dim
        
        self.Wq = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.Wk = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.Wv = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
    
    def forward(self, token_encodings, mask):
        q = self.Wq(token_encodings)
        k = self.Wk(token_encodings)
        v = self.Wv(token_encodings)

        sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))
        scaled_sims = sims/torch.tensor(k.size(self.col_dim) ** 0.5)
        scaled_sims = scaled_sims + mask
        attention_percents = F.softmax(scaled_sims, dim=self.col_dim) ### determines the percentage of influence each token should have on each other
        attention_scores = torch.matmul(attention_percents, v)
        return attention_scores
        

In [39]:
encodings_matrix = torch.tensor([[1.16, 0.23],
                                 [0.57, 1.36],
                                 [4.41, -2.16]])

torch.manual_seed(42)
selfAttention = SelfAttention()

In [40]:
mask = torch.tril(torch.ones(3, 3))
mask = torch.triu(torch.ones(3, 3), diagonal=1) * -1e9
mask

tensor([[-0.0000e+00, -1.0000e+09, -1.0000e+09],
        [-0.0000e+00, -0.0000e+00, -1.0000e+09],
        [-0.0000e+00, -0.0000e+00, -0.0000e+00]])

In [41]:
selfAttention.forward(encodings_matrix, mask)

tensor([[ 0.6038,  0.7434],
        [-0.0062,  0.6072],
        [ 3.4989,  2.2427]], grad_fn=<MmBackward0>)

In [42]:
# dmodel is the size of the weight matrices
# if dmodel is 2 we have a 2 word embedding value

### This is encoder decoder attention
class Attention(nn.Module):
    def __init__(self, d_model=2, row_dim=0, col_dim=1):
        super().__init__()
        self.d_model = d_model
        self.row_dim = row_dim
        self.col_dim = col_dim
        
        self.Wq = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.Wk = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.Wv = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
    
    def forward(self, encoding_q, encoding_k, encoding_v, mask=None):
        q = self.Wq(encoding_q)
        k = self.Wk(encoding_k)
        v = self.Wv(encoding_v)

        sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))
        scaled_sims = sims/torch.tensor(k.size(self.col_dim) ** 0.5)
        if mask != None:
            scaled_sims = scaled_sims + mask
        
        attention_percents = F.softmax(scaled_sims, dim=self.col_dim) ### determines the percentage of influence each token should have on each other
        attention_scores = torch.matmul(attention_percents, v)
        return attention_scores
        

In [43]:
encodings_q = torch.tensor([[1.16, 0.23],
                                 [0.57, 1.36],
                                 [4.41, -2.16]])
encodings_k = torch.tensor([[1.16, 0.23],
                                 [0.57, 1.36],
                                 [4.41, -2.16]])
encodings_v = torch.tensor([[1.16, 0.23],
                                 [0.57, 1.36],
                                 [4.41, -2.16]])
mask = torch.tril(torch.ones(3, 3))
mask = torch.triu(torch.ones(3, 3), diagonal=1) * -1e9
attention = Attention()

In [44]:
attention.forward(encodings_q, encodings_k, encodings_v)

tensor([[1.0100, 1.0641],
        [0.2040, 0.7057],
        [3.4989, 2.2427]], grad_fn=<MmBackward0>)

In [45]:


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=2, row_dim=0, col_dim=1, num_heads=1):
        super().__init__()
        self.heads = nn.ModuleList([Attention(d_model, row_dim, col_dim) for _ in range(num_heads)])
        self.col_dim = col_dim
    
    def forward(self, encodings_q, encodings_k, encodings_v):
        probs = torch.cat([head(encodings_q, encodings_k, encodings_v) for head in self.heads], dim=self.col_dim)
        return probs

In [57]:
torch.manual_seed(42)
encodings_q = torch.tensor([[1.16, 0.23],
                                 [0.57, 1.36],
                                 [4.41, -2.16]])
encodings_k = torch.tensor([[1.16, 0.23],
                                 [0.57, 1.36],
                                 [4.41, -2.16]])
encodings_v = torch.tensor([[1.16, 0.23],
                                 [0.57, 1.36],
                                 [4.41, -2.16]])
mh = MultiHeadAttention(num_heads=8)
mh(encodings_q, encodings_k, encodings_v)

tensor([[ 1.0100,  1.0641, -0.7081, -0.8268,  0.6226,  0.1312,  1.0106,  0.8625,
          0.3422,  0.7333, -0.8037,  1.4087, -0.6674,  0.5665,  0.7700, -0.9269],
        [ 0.2040,  0.7057, -0.7417, -0.9193,  0.5522,  0.2499,  1.4153,  1.0420,
          0.6753,  2.1341, -0.7498,  0.9677, -0.5970,  1.5640,  0.7713, -0.9210],
        [ 3.4989,  2.2427, -0.7190, -0.8447,  0.5669,  0.2324,  0.3679,  0.5894,
          0.1412, -0.1826, -0.9414,  2.2589, -0.7832, -0.0405,  0.7669, -0.8751]],
       grad_fn=<CatBackward0>)